# Evaluation Notebook — Adaptive Anti-Doping Defense Engine

Skeleton for dataset validation and detector evaluation.

| Section | Status |
|---|---|
| 1. Setup / imports | Functional (Day 1) |
| 2. Load data | Functional (Day 1) |
| 3. Basic schema / sanity checks | Functional (Day 1) |
| 4. Visual trajectory checks | Stub — Day 2 |
| 5. Anomaly stats | Stub — Day 3+ |
| 6. Precision / recall vs ground truth | Stub — Day 4+ |

## 1. Setup / imports

In [1]:
from __future__ import annotations

import json
import math
from datetime import date
from pathlib import Path

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"

ATHLETE_FIELDS = ("id", "name", "sport", "age", "baseline_prior_json")
SAMPLE_FIELDS = (
    "id",
    "athlete_id",
    "date",
    "hb",
    "hct",
    "ret_pct",
    "off_score",
    "te_ratio",
    "competition_flag",
    "altitude_flag",
    "injury_flag",
)

HB_MIN, HB_MAX = 12.0, 18.0


def compute_off_score(hb_g_dL: float, ret_pct: float) -> float:
    return (hb_g_dL * 10) - (60 * math.sqrt(ret_pct))


print(f"Repo root: {REPO_ROOT}")
print(f"Data dir:  {DATA_DIR}")

Repo root: C:\Users\srish\OneDrive\Desktop\Projects\Anti-Doping-System
Data dir:  C:\Users\srish\OneDrive\Desktop\Projects\Anti-Doping-System\data


## 2. Load data

In [2]:
athletes_path = DATA_DIR / "athletes.json"
samples_path = DATA_DIR / "samples.json"

with athletes_path.open(encoding="utf-8") as f:
    athletes = json.load(f)
with samples_path.open(encoding="utf-8") as f:
    samples = json.load(f)

print(f"Loaded {len(athletes)} athletes from {athletes_path.name}")
print(f"Loaded {len(samples)} samples from {samples_path.name}")

Loaded 10 athletes from athletes.json
Loaded 50 samples from samples.json


## 3. Basic schema / sanity checks

In [3]:
def is_nan(value) -> bool:
    return isinstance(value, float) and math.isnan(value)


def run_sanity_checks(athletes: list[dict], samples: list[dict]) -> tuple[bool, list[str]]:
    errors: list[str] = []

    athlete_ids = set()
    for athlete in athletes:
        for field in ATHLETE_FIELDS:
            if field not in athlete:
                errors.append(f"Athlete missing field '{field}': {athlete}")

        if not isinstance(athlete.get("id"), int):
            errors.append(f"Athlete id must be int, got {type(athlete.get('id'))}")
        if not isinstance(athlete.get("name"), str):
            errors.append(f"Athlete {athlete.get('id')}: name must be str")
        if not isinstance(athlete.get("sport"), str):
            errors.append(f"Athlete {athlete.get('id')}: sport must be str")
        if athlete.get("age") is not None and not isinstance(athlete.get("age"), int):
            errors.append(f"Athlete {athlete.get('id')}: age must be int or null")

        athlete_ids.add(athlete["id"])

    sample_ids = set()
    for sample in samples:
        for field in SAMPLE_FIELDS:
            if field not in sample:
                errors.append(f"Sample {sample.get('id')}: missing field '{field}'")

        sid = sample.get("id")
        if not isinstance(sid, int):
            errors.append(f"Sample id must be int, got {type(sid)}")
        elif sid in sample_ids:
            errors.append(f"Duplicate sample id: {sid}")
        else:
            sample_ids.add(sid)

        if sample.get("athlete_id") not in athlete_ids:
            errors.append(
                f"Sample {sample.get('id')}: invalid athlete_id {sample.get('athlete_id')}"
            )

        try:
            date.fromisoformat(sample["date"])
        except (TypeError, ValueError):
            errors.append(f"Sample {sample.get('id')}: invalid date '{sample.get('date')}'")

        hb = sample.get("hb")
        if hb is None or is_nan(hb):
            errors.append(f"Sample {sample.get('id')}: hb is NaN or missing")
        elif not (HB_MIN <= hb <= HB_MAX):
            errors.append(f"Sample {sample.get('id')}: hb={hb} outside [{HB_MIN}, {HB_MAX}]")

        for field in ("hct", "ret_pct", "off_score", "te_ratio"):
            val = sample.get(field)
            if val is None or is_nan(val):
                errors.append(f"Sample {sample.get('id')}: {field} is NaN or missing")

        for flag in ("competition_flag", "altitude_flag", "injury_flag"):
            if sample.get(flag) not in (True, False):
                errors.append(f"Sample {sample.get('id')}: {flag} must be bool")

        expected_off = round(compute_off_score(sample["hb"], sample["ret_pct"]), 1)
        if sample["off_score"] != expected_off:
            errors.append(
                f"Sample {sample.get('id')}: off_score {sample['off_score']} != expected {expected_off}"
            )

    return len(errors) == 0, errors


passed, sanity_errors = run_sanity_checks(athletes, samples)

print("=" * 60)
print("SANITY CHECK SUMMARY")
print("=" * 60)
print(f"Athletes: {len(athletes)}")
print(f"Samples:  {len(samples)}")
print(f"Hb range: {min(s['hb'] for s in samples):.2f} - {max(s['hb'] for s in samples):.2f} g/dL")
print(f"Result:   {'PASS' if passed else 'FAIL'}")
if sanity_errors:
    print("\nErrors:")
    for err in sanity_errors:
        print(f"  - {err}")

SANITY CHECK SUMMARY
Athletes: 10
Samples:  50
Hb range: 13.57 - 16.60 g/dL
Result:   PASS


## 4. Visual trajectory checks

> Stub — real plotting work lands Day 2 (v1.1 data with anomaly injection).

In [4]:
# TODO (Day 2): plot 3-4 athlete Hb trajectories; overlay normal vs anomalous series.
pass

## 5. Anomaly stats

> Stub — requires anomaly detector + `anomalies` table (Day 3+).

In [5]:
# TODO (Day 3+): compute distribution of anomaly scores, flag rates, etc.
pass

## 6. Precision / recall vs ground truth

> Stub — requires hidden `ground_truth.json` + frozen detector (Day 4+).

In [6]:
# TODO (Day 4+): load ground_truth.json, compute precision/recall at chosen threshold.
pass